# Live Quantum Hardware QRNG Demo
### AstraCode Qiskit Fall Fest 2026 · Project 4: Quantum Random Number Generator

This notebook connects to real IBM Quantum superconducting hardware to submit an asynchronous 8-qubit Hadamard circuit, check the queue status non-blockingly, and save verified true-random bits to `data/live_runs/`.

> **Note:** Our verified baseline dataset (`data/quantum_bits_hardware.json`, from `ibm_marrakesh`, job `dbplvroqtr0c738elbe0`) is protected and never overwritten. Live runs are saved exclusively into timestamped archives and `data/live_runs/latest.json`.


### Step 1: Authentication and Environment Setup
We load required modules and verify IBM Quantum credentials. If `IBM_QUANTUM_TOKEN` is already configured in your environment, entering your key is automatically skipped. Your API key is never saved or printed.


In [ ]:
import getpass
import os
from pathlib import Path
import matplotlib.pyplot as plt

import qiskit
from qiskit_ibm_runtime import QiskitRuntimeService

from quantum_engine import (
    build_qrng_circuit,
    submit_hardware_job,
    save_live_run,
    bits_to_numbers,
    DEFAULT_QUBITS,
    DEFAULT_N_BITS,
)

# Authentication: prompt only if not already in environment
if "IBM_QUANTUM_TOKEN" not in os.environ:
    token_val = getpass.getpass("Enter your IBM Quantum API token: ").strip()
    if token_val:
        os.environ["IBM_QUANTUM_TOKEN"] = token_val

if "IBM_QUANTUM_INSTANCE" not in os.environ:
    inst_val = getpass.getpass("Enter IBM instance CRN or name (press Enter to skip): ").strip()
    if inst_val:
        os.environ["IBM_QUANTUM_INSTANCE"] = inst_val

if os.environ.get("IBM_QUANTUM_TOKEN"):
    print("IBM Quantum authentication credentials loaded successfully (hidden).")
else:
    print("Notice: No IBM Quantum token set. Run this cell with your API key to submit live hardware jobs.")


### Step 2: The 8-Qubit QRNG Circuit
We construct our quantum circuit using `build_qrng_circuit(n_qubits=8)`. Applying a Hadamard ($H$) gate to each qubit maps $|0\rangle \to \frac{|0\rangle + |1\rangle}{\sqrt{2}}$, creating an equal 50/50 superposition. Measurement collapses each wavefunction into a truly unpredictable classical bit (0 or 1).


In [ ]:
qc = build_qrng_circuit(n_qubits=8)
print(f"Circuit created: {qc.num_qubits} qubits, {qc.num_clbits} classical bits.")
display(qc.draw("mpl"))


### Step 3: Submit Live Hardware Job (Non-blocking)
We find the least busy operational IBM Quantum physical processor and submit 125 shots (125 shots $\times$ 8 qubits = 1,000 bits).
Crucially, this submission is **asynchronous** and **non-blocking**: it returns immediately so you can continue your presentation or analysis without freezing the notebook.


In [ ]:
token = os.environ.get("IBM_QUANTUM_TOKEN")
if not token:
    print("IBM_QUANTUM_TOKEN is not set. Please enter your API token in Step 1 first.")
else:
    print("Selecting least-busy real QPU and submitting job...")
    job, backend_name = submit_hardware_job(qc, shots=125)
    print(f"Target Backend: {backend_name}")
    print(f"Job ID:         {job.job_id()}")
    st = job.status()
    status_name = (st.name if hasattr(st, "name") else str(st)).upper()
    print(f"Initial Status: {status_name}")
    print("Job submitted successfully! Proceed to Step 4 to monitor progress.")


### Step 4: Check Job Status & Queue Position (Re-runnable)
You can re-run this cell at any time to inspect the job status and queue position. If the queue wait exceeds your timeout setting (default 10 minutes), the cell will notify you so you can seamlessly fall back to presenting our verified saved hardware baseline.


In [ ]:
TIMEOUT_MINUTES = 10
TIMEOUT_SECONDS = TIMEOUT_MINUTES * 60

if "job" not in globals():
    print("No active live job submitted yet. Run Step 3 first.")
else:
    st = job.status()
    status_name = (st.name if hasattr(st, "name") else str(st)).upper()
    backend_label = globals().get("backend_name") or (job.backend().name if hasattr(job, "backend") and hasattr(job.backend(), "name") else "ibm_qpu")
    print(f"Backend: {backend_label}")
    print(f"Job ID:  {job.job_id()}")
    print(f"Status:  {status_name}")
    
    # Check queue position if available
    try:
        q_pos = getattr(job, "queue_position", None)
        if callable(q_pos):
            pos_val = q_pos()
            if pos_val is not None:
                print(f"Queue Position: {pos_val}")
    except Exception:
        pass
    
    if status_name == "DONE":
        print("\nJob complete! Run Step 5 below to extract, visualize, and save the live bits.")
    elif status_name in ("QUEUED", "RUNNING"):
        print(f"\nJob is currently {status_name}. You can re-run this cell whenever you want an update.")
        print(f"If the queue wait exceeds {TIMEOUT_MINUTES} minutes, you can switch to our saved baseline.")
    elif status_name in ("ERROR", "CANCELLED"):
        print(f"\nJob ended with {status_name}. Please check your IBM Quantum dashboard or use the saved run.")


### Step 5: Extract, Inspect, and Save Live Quantum Results
Once the job completes, we retrieve the measurements. In accordance with Qiskit's little-endian bitstring convention, each shot bitstring is reversed (`[::-1]`) to align bit index $i$ with qubit $i$.
The results are saved **only** to `data/live_runs/live_<backend>_<timestamp>.json` and `data/live_runs/latest.json`. Any attempt to overwrite the shared baseline `quantum_bits_hardware.json` is strictly blocked.


In [ ]:
if "job" not in globals():
    print("No job submitted. Run Step 3 first.")
else:
    # Retrieve job result
    st = job.status()
    status_name = (st.name if hasattr(st, "name") else str(st)).upper()
    backend_label = globals().get("backend_name") or (job.backend().name if hasattr(job, "backend") and hasattr(job.backend(), "name") else "ibm_qpu")
    if status_name != "DONE":
        print("Waiting for job completion (up to timeout)...")
        result = job.result(timeout=TIMEOUT_SECONDS)
    else:
        result = job.result()
    
    # Extract bitstrings with little-endian reversal
    raw_bitstrings = result[0].data.c.get_bitstrings()
    bits = []
    for shot in raw_bitstrings:
        for char in shot[::-1]:
            bits.append(int(char))
            if len(bits) == 1000:
                break
        if len(bits) == 1000:
            break
            
    # Convert into 8-bit integers (0 to 255)
    numbers = bits_to_numbers(bits, n_qubits=8)
    
    ones_count = sum(bits)
    zeros_count = len(bits) - ones_count
    fraction_ones = ones_count / len(bits)
    
    print(f"Successfully collected {len(bits)} bits from {backend_label} (Job: {job.job_id()})")
    print(f"Ones: {ones_count} | Zeros: {zeros_count} | Fraction of 1s: {fraction_ones:.4f}")
    print(f"First 32 bits: {bits[:32]}")
    print(f"First 10 numbers (0-255): {numbers[:10]}")
    
    # Plot frequency bar chart
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bars = ax.bar(["0", "1"], [zeros_count, ones_count], color=["#4f46e5", "#10b981"], width=0.4)
    ax.set_ylabel("Measured Count")
    ax.set_title(f"Live Measured Bit Frequencies on {backend_label}")
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 10, str(h), ha="center", fontweight="bold")
    ax.set_ylim(0, max(zeros_count, ones_count) + 80)
    plt.tight_layout()
    plt.show()
    
    # Save safely to data/live_runs/
    run_file, latest_file = save_live_run(bits, backend=backend_label, job_id=job.job_id(), n_qubits=8)
    print(f"Archived timestamped run: {run_file}")
    print(f"Refreshed latest pointer: {latest_file}")
    print("\nYou can now open 'qrng_final.ipynb' with DATASET = 'live' to analyze this fresh run!")


## Fallback: Verified Hardware Baseline Run

If the physical hardware queue is busy during your demo, you do not need to wait. Our verified shared baseline is always available and pre-computed:
- **Processor:** `ibm_marrakesh`
- **Job ID:** `dbplvroqtr0c738elbe0`
- **Measured Balance:** 512 ones / 1,000 bits (51.2% ones, $p = 0.447884$)
- **IBM Quantum Jobs Dashboard:** You can view the live execution record for this job anytime at [IBM Quantum Jobs Dashboard](https://quantum.ibm.com/jobs) by searching for job ID `dbplvroqtr0c738elbe0`.
